# TCGA-KIRC — R934.3: Auditoria de metadados GDC corrigida (somente metadados)

**Versão corretiva R934.3**: conserva o contrato R934, corrige a checagem de schema files.* versus projeções cases.*, valida releases pela interseção de `latest_release`, aplica somente a exclusão workflow autorizada e grava uma execução UTC versionada no Drive.
O endpoint `/files/_mapping` valida campos indexados de arquivos; os campos associados a `cases.*` são solicitados e verificados na resposta real de `/files`.
A falha anterior `GateBlocked: campos não confirmados no schema GDC` não comprova falta de campos nos registros.

**Operação permitida**: consulta metadata-only a arquivos open-access, nenhum download de RNA-seq, nenhuma execução do pipeline estatístico.
A consulta P1 nesta versão é metadata-only. O relatório público será agregado; `private_metadata` contém relações file/patient e não entra no ZIP compartilhável.


### Mapa das etapas e controles

| Etapa | Implementação | Regra de parada |
|---|---|---|
| P0 | especificação congelada, testes positivos e contraprovas, aceite metodológico explícito | sem aceite, não se passa a P1 aprovado |
| P1 | GDC status/schema, metadados estáveis em duas passagens, exclusão autorizada, latest_release comum e orçamento ≤ 6 GiB | falha em acesso, checksum, workflow, interseção release, origem amostral ou orçamento |
| P2 | downloader verificável **opcional** por MD5/SHA-256 e limite total | proíbe acesso controlado e arquivo não manifestado |
| P3 | split por paciente e testes de normalização/calibração em cenário sintético | **não libera modelagem prognóstica real** sem pipeline nested-CV/VST validado |
| P4 | estrutura de registro de AUC/IC/Brier/calibração, holdout único | permanece pendente até execução e revisão independentes |

**Importante:** o R934 anterior relata consulta *metadata-only* em 21/10/2026, 614 IDs únicos e 605 selecionados antes da exclusão autorizada, mas o manifesto precisa ser refeito nesta política. As contagens anteriores não se promovem automaticamente a P1 GREEN.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
import json, hashlib, re, os, statistics, unittest, io, contextlib, shutil, uuid
from urllib.parse import urlparse
from collections import Counter

# Autorização explícita do operador: persistir resultados no Drive via runtime Colab.
MONTAR_DRIVE = True
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('R934.3 deve rodar em um runtime Google Colab autorizado.') from exc
if MONTAR_DRIVE and not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/R934_ccRCC')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
ROOT = PROJECT_ROOT / 'runs' / RUN_ID
ROOT.mkdir(parents=True, exist_ok=False)  # diretório novo; nunca sobrescrever uma execução anterior
for name in ('private_metadata','logs','reports','data_open','tests','models'):
    (ROOT/name).mkdir(parents=True, exist_ok=True)
PUBLIC_DIRS=['tests','reports','logs']  # private_metadata e data_open nunca entram no ZIP público
def timestamp(): return datetime.now(timezone.utc).isoformat()
def sha256_obj(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()
def write_json(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,ensure_ascii=False,default=str),encoding='utf8')
    tmp.replace(path)
    return path
def record_stage(stage,status,details=None):
    row={'timestamp_utc':timestamp(),'run_id':RUN_ID,'stage':stage,'status':status,'details':details or {}}
    with (ROOT/'logs'/'stage_events.jsonl').open('a',encoding='utf8') as out: out.write(json.dumps(row,ensure_ascii=False,sort_keys=True)+'\n')
RUN_METADATA={'run_id':RUN_ID,'started_at_utc':timestamp(),'output_root':str(ROOT),'drive_mounted':MONTAR_DRIVE,'file_content_downloaded':False,'analysis_executed':False}
write_json(ROOT/'reports'/'run_metadata.json',RUN_METADATA)
record_stage('runtime_start','initialized',{'drive_run_directory':str(ROOT)})
print('Run UTC versionado no Drive:',ROOT)

## 1. Política de execução e aprovação explícita

Os parâmetros abaixo **não autorizam baixar dados**. P0 requer testes GREEN, escopo/metodologia congelados e revisão/aceite independente registrado. P1 exige manifesto reproduzível. `CONSULTAR_GDC_METADATA=True` nesta R934.3 envia somente requisições metadata-only; `EXECUTAR_DOWNLOADS=False` é bloqueio fixo.

In [ ]:
PROJECT='TCGA-KIRC'
SEED=2026
GDC='https://api.gdc.cancer.gov'
MAX_AGGREGATE_BYTES=6*1024**3
REQUIRE_OPEN=True
REQUIRE_RELEASED=True
CONSULTAR_GDC_METADATA=True    # R934.2: solicitado prosseguir com METADADOS, SEM conteúdo de RNA-seq
EXECUTAR_DOWNLOADS=False       # HARD BLOCK: nenhum download neste caderno
ACEITE_METODOLOGICO=''         # quando aplicável: preencher identificação/revisão e registrar em arquivo independente
CONFIRMACAO_DOWNLOAD=''       # para transferir: inserir exatamente 'AUTORIZO_DOWNLOAD_OPEN_6GIB'
EXPECTED_WORKFLOW='STAR - Counts'

GLYCOLYSIS=['SLC2A1','SLC2A3','HK1','HK2','HK3','GPI','PFKP','PFKM','PFKL','ALDOA','ALDOC','TPI1','GAPDH','PGK1','PGAM1','ENO1','ENO2','PKM','LDHA','PDK1','PDK3','SLC16A3']
TCA_OXPHOS=['PDHA1','PDHB','DLAT','DLD','CS','ACO2','IDH1','IDH2','IDH3A','OGDH','DLST','SUCLA2','SUCLG1','SDHA','SDHB','SDHC','SDHD','FH','MDH1','MDH2','PPARGC1A']
GLUCONEOGENESIS=['PCK1','PCK2']  # não confundir com TCA/OXPHOS
PANEL_APPROVED=False  # lista candidata requer conferência de fontes biológicas antes de congelar

SPEC=dict(project=PROJECT,seed=SEED,holdout_fraction=.30,primary_horizon_months=60,
          budget_bytes=MAX_AGGREGATE_BYTES,access='open',state='released',
          workflow=EXPECTED_WORKFLOW,outer_folds=5,outer_repeats=5,inner_folds=5,
          final_lambda='lambda.1se',sensitivity_lambda='lambda.min',
          auc_bootstrap=1000,min_bootstrap_convergence=.90,
          min_events_by_60m_holdout=10,min_censor_survival_at_60m=.10,
          glycolysis=GLYCOLYSIS,tca_oxphos=TCA_OXPHOS,gluconeogenesis=GLUCONEOGENESIS,
          auc_metric='cumulative_dynamic_IPCW',missingness_limit_per_covariate=.10)
SPEC_HASH=sha256_obj(SPEC)
write_json(ROOT/'reports'/'spec_R934_1.json',{'spec':SPEC,'sha256':SPEC_HASH,'saved_at_utc':timestamp()})
print('Hash do contrato SDD:',SPEC_HASH)

## 2. TDD — contraprovas RED→GREEN (inteiramente sintético, sem pacotes externos)

TDD **não avalia** se AUC, HR, MUI ou p-valor têm direção favorável. Os testes abaixo detectam erros de integridade. A função `reject()` lança um erro quando uma invariável é violada; contraprova RED é aprovada quando o erro esperado ocorre e a amostra positiva passa.

In [ ]:
class GateBlocked(RuntimeError): pass
class MetadataError(GateBlocked): pass

def require(condition,msg):
    if not condition: raise GateBlocked(msg)

def validate_counts(matrix):
    # matriz Python: genes x amostras; só TDD com valores sintéticos
    require(bool(matrix) and bool(matrix[0]),'matriz vazia')
    width=len(matrix[0]);require(all(len(row)==width for row in matrix),'dimensões irregulares')
    for row in matrix:
        for x in row:
            require(isinstance(x,(int,float)) and x>=0 and x!=float('inf') and x==x and float(x).is_integer(),
                    'contagens devem ser inteiros >= 0, finitos')
    require(all(sum(row[j] for row in matrix)>0 for j in range(width)),'biblioteca zerada')
    return True

def validate_patient_partition(train_ids,test_ids):
    require(len(train_ids)==len(set(train_ids)),'duplicatas em discovery')
    require(len(test_ids)==len(set(test_ids)),'duplicatas em holdout')
    require(not (set(train_ids)&set(test_ids)),'leakage: paciente em discovery e holdout')
    return True

def fit_train_scaler(x_train):
    # prova de contrato, não substitui VST DESeq2
    require(x_train and len(x_train)>1,'treino insuficiente')
    p=len(x_train[0]);require(all(len(x)==p for x in x_train),'dimensões inválidas')
    mu=[statistics.mean(col) for col in zip(*x_train)]
    sd=[statistics.stdev(col) or 1.0 for col in zip(*x_train)]
    return mu,sd

def apply_frozen_scaler(x,mu,sd):
    require(all(len(row)==len(mu) for row in x),'features incompatíveis')
    return [[(v-m)/s for v,m,s in zip(row,mu,sd)] for row in x]

def validate_budget(files,max_bytes=MAX_AGGREGATE_BYTES):
    ids=[r['file_id'] for r in files]
    require(len(ids)==len(set(ids)),'file_id duplicado; somatório ambíguo')
    require(all(r.get('access')=='open' and r.get('state')=='released' for r in files),'acesso controlado ou estado não-released')
    require(all(type(r.get('file_size')) is int and r['file_size']>0 for r in files),'file_size inválido')
    total=sum(r['file_size'] for r in files)
    require(total<=max_bytes,f'orçamento excedido: {total} > {max_bytes}')
    return total

def expect_red(fn,*args,**kwargs):
    try: fn(*args,**kwargs)
    except GateBlocked: return True
    return False

class R934SyntheticTDD(unittest.TestCase):
    def test_counts_green(self): self.assertTrue(validate_counts([[1,2],[3,1]]))
    def test_counts_nan_red(self): self.assertTrue(expect_red(validate_counts,[[1,float('nan')],[2,1]]))
    def test_counts_fraction_red(self): self.assertTrue(expect_red(validate_counts,[[1,1.5],[2,1]]))
    def test_zero_library_red(self): self.assertTrue(expect_red(validate_counts,[[1,0],[2,0]]))
    def test_patient_isolation_green(self): self.assertTrue(validate_patient_partition(['P1','P2'],['P3']))
    def test_patient_leakage_red(self): self.assertTrue(expect_red(validate_patient_partition,['P1','P2'],['P2','P3']))
    def test_frozen_parameters_green(self):
        mu,sd=fit_train_scaler([[1,10],[3,14],[5,18]])
        z=apply_frozen_scaler([[101,14]],mu,sd)
        self.assertAlmostEqual(z[0][0],(101-mu[0])/sd[0]); self.assertEqual(mu[0],3)
    def test_controlled_access_red(self):
        self.assertTrue(expect_red(validate_budget,[{'file_id':'A','access':'controlled','state':'released','file_size':10}]))
    def test_over_budget_red(self):
        self.assertTrue(expect_red(validate_budget,[{'file_id':'A','access':'open','state':'released','file_size':MAX_AGGREGATE_BYTES+1}]))
    def test_duplicate_file_red(self):
        f=[{'file_id':'X','access':'open','state':'released','file_size':100}]
        self.assertTrue(expect_red(validate_budget,f+f))
    def test_budget_green(self):
        self.assertEqual(validate_budget([{'file_id':'X','access':'open','state':'released','file_size':500}]),500)

buffer=io.StringIO()
suite=unittest.defaultTestLoader.loadTestsFromTestCase(R934SyntheticTDD)
result=unittest.TextTestRunner(stream=buffer,verbosity=2).run(suite)
TDD_GREEN = result.wasSuccessful()
write_json(ROOT/'tests'/'tdd_report.json',{'timestamp_utc':timestamp(),'tests':result.testsRun,
    'failures':[str(e) for _,e in result.failures], 'errors':[str(e) for _,e in result.errors],
    'green':TDD_GREEN,'scope':'synthetic preflight only','log':buffer.getvalue()})
print(buffer.getvalue())
require(TDD_GREEN,'P0 sintético: testes falharam')

## 3. Contrato de seleção do GDC — exclusivamente metadata-only

Consulta GET ao `https://api.gdc.cancer.gov/status`, GET ao `/files/_mapping` e paginação GET `/files` (não `/data`). No Colab, execute esta seção **somente se desejar atualização de metadados**. A enumeração anterior de 614/605 não é uma validação reproduzida pelo notebook.

O GDC pode mudar seu esquema; se um campo ou filtro falhar, **parar**, registrar e revisar. Não remover o filtro `access=open` para prosseguir. Os metadados brutos ficam em `private_metadata/` (não compartilhar). Acesso a dados individuais continua proibido.

In [ ]:
GDC_FIELDS=[
 'file_id','file_name','file_size','md5sum','access','state','data_category','data_type',
 'analysis.workflow_type','analysis.workflow_version',
 'cases.case_id','cases.submitter_id','cases.project.project_id',
 'cases.samples.sample_id','cases.samples.submitter_id','cases.samples.sample_type',
 'cases.samples.portions.analytes.aliquots.aliquot_id',
 'cases.samples.portions.analytes.aliquots.submitter_id']
GDC_FILTER={'op':'and','content':[
 {'op':'=','content':{'field':'cases.project.project_id','value':PROJECT}},
 {'op':'=','content':{'field':'files.data_category','value':'Transcriptome Profiling'}},
 {'op':'=','content':{'field':'files.data_type','value':'Gene Expression Quantification'}},
 {'op':'=','content':{'field':'files.analysis.workflow_type','value':EXPECTED_WORKFLOW}},
 {'op':'=','content':{'field':'files.access','value':'open'}},
 {'op':'=','content':{'field':'files.state','value':'released'}}]}

def gdc_get(path,params=None):
    import requests
    # Endpoint fixo, HTTPS, sem redirecionamento ou fallback para controlled-access.
    require(isinstance(path,str) and (path in {'/status','/files','/files/_mapping'} or path.startswith('/files/versions/')),'endpoint GDC não permitido')
    url=GDC+path; parsed=urlparse(url)
    require(parsed.scheme=='https' and parsed.netloc=='api.gdc.cancer.gov','host GDC inválido')
    response=requests.get(url,params=params,timeout=90,headers={'Accept':'application/json'},allow_redirects=False)
    require(200<=response.status_code<300,f'redirecionamento/HTTP GDC bloqueado: {response.status_code}')
    require(response.url==response.request.url,'URL GDC mudou após requisição')
    require(len(response.content)<=8*1024*1024,'resposta metadata excede 8 MiB')
    payload=response.json()
    if path.startswith('/files/versions/'):
        require(isinstance(payload,list),'resposta /files/versions não é lista JSON')
        return payload
    require(isinstance(payload,dict),'resposta GDC não é JSON-objeto')
    if 'errors' in payload: raise MetadataError(f'GDC retornou erros: {payload["errors"]}')
    if path=='/files':
        warnings=payload.get('warnings')
        require(isinstance(warnings,dict),'warnings ausentes/malformados no /files')
        require(not warnings,f'GDC /files warnings: {warnings}')
    return payload

def verify_files_schema(mapping):
    """Check the files index only. Related case fields are NOT necessarily indexed here.

    Documentation for /files confirms that case/sample/aliquot fields can be
    retrieved as related fields without being listed in /files/_mapping.
    Verify those projections against actual /files metadata responses, not this map.
    """
    m=mapping.get('_mapping',{})
    require(isinstance(m,dict) and bool(m),'schema /files/_mapping ausente')
    required_indexed={
       'files.file_id','files.file_name','files.access','files.state','files.data_category','files.data_type',
       'files.analysis.workflow_type','files.analysis.workflow_version',
       'files.file_size','files.md5sum'
    }
    missing=sorted(field for field in required_indexed if field not in m)
    require(not missing,'campos do índice files não confirmados no schema GDC: '+', '.join(missing))
    return True

def metadata_pages(page_size=200,pass_label='pass1'):
    require(0<page_size<=1000,'page_size inválido')
    status=gdc_get('/status')
    mapping=gdc_get('/files/_mapping')
    verify_files_schema(mapping)
    # Map é arquivado integralmente e com hash para auditoria; não inferir campos faltantes.
    q={'filters':json.dumps(GDC_FILTER,separators=(',',':')),
       'fields':','.join(GDC_FIELDS),'format':'JSON','size':page_size,'from':0,'sort':'file_id:asc'}
    rows=[]; page_audit=[]; expected_total=None; page_no=0
    raw_dir=ROOT/'private_metadata'/pass_label; raw_dir.mkdir(parents=True,exist_ok=True)
    while True:
        q['from']=len(rows)
        q['size']=page_size if expected_total is None else min(page_size,expected_total-len(rows))
        response=gdc_get('/files',dict(q))
        data=response.get('data',{})
        hits=data.get('hits')
        pag=data.get('pagination',{})
        require(isinstance(hits,list) and all(type(pag.get(k)) is int for k in ('total','from','size','count')),
                'schema de paginação incompatível')
        if expected_total is None: expected_total=pag['total']
        require(pag['from']==q['from'] and pag['size']==q['size'] and pag['count']==len(hits),
                'offset/size/count de paginação divergente')
        # Related cases.* fields are checked on the actual returned JSON.
        # A valid mapping of files.* alone is not proof those relations were retrieved.
        for hit in hits:
            require(isinstance(hit,dict),'arquivo GDC não é objeto JSON')
            missing_file=[k for k in ('file_id','access','state','file_size','md5sum','analysis','cases')
                          if k not in hit]
            require(not missing_file,'campos obrigatórios ausentes no retorno /files: '+', '.join(missing_file))
            require(isinstance(hit.get('cases'),list),'/files: cases deve ser uma lista')
            require(hit['cases'] and all(isinstance(c,dict) for c in hit['cases']),
                    '/files: vínculo de caso ausente/inválido')
        require(pag['total']==expected_total,'total de paginação mudou durante a consulta')
        require(len(hits)>0 or len(rows)==expected_total,'página vazia antes do total')
        page_no+=1
        page_ids=[str(hit.get('file_id',hit.get('id',''))) for hit in hits]
        require(page_ids==sorted(page_ids),'ordem file_id não monotônica na página')
        write_json(raw_dir/f'gdc_raw_page_{page_no:04d}.json', response)
        page_audit.append({'page':page_no,'from':q['from'],'size':len(hits),'sha256':sha256_obj(response)})
        rows.extend(hits)
        require(len(rows)<=expected_total,'mais registros do que o total GDC')
        if len(rows)==expected_total: break
    ids=[r.get('file_id',r.get('id')) for r in rows]
    require(len(ids)==len(set(ids)),'file_id duplicados no retorno da API; consultar novamente')
    require(None not in ids,'file_id ausente')
    require(ids==sorted(ids),'file_id não monotônico entre páginas')
    metadata={'retrieved_utc':timestamp(),'pass_label':pass_label,'api':GDC,'status':status,
      'query_filter':GDC_FILTER,'fields_requested':GDC_FIELDS,
      'sort':'file_id:asc','total_api':expected_total,'pages':page_audit,'schema_sha256':sha256_obj(mapping),
      'query_sha256':sha256_obj(GDC_FILTER),'records_sha256':sha256_obj(rows)}
    write_json(ROOT/'private_metadata'/f'gdc-files-metadata-raw-{pass_label}.json',{'audit':metadata,'files':rows})
    write_json(ROOT/'private_metadata'/f'gdc-schema-mapping-{pass_label}.json',mapping)
    return metadata,rows


In [ ]:
VERSION_BATCH_SIZE=50
RELEASE_RE=re.compile(r'^[0-9]+\.[0-9]+$')
def _release_key(value): return tuple(int(part) for part in value.split('.'))

def fetch_latest_versions(file_ids):
    require(file_ids and len(file_ids)<=VERSION_BATCH_SIZE,'lote /files/versions deve conter 1..50 UUIDs')
    require(all(UUID_RE.fullmatch(str(fid)) for fid in file_ids),'UUID inválido em /files/versions')
    payload=gdc_get('/files/versions/'+','.join(file_ids))
    require(isinstance(payload,list),'resposta /files/versions não é lista')
    result={}
    for item in payload:
        require(isinstance(item,dict) and isinstance(item.get('id'),str),'registro de versão malformado')
        require(item['id'] in file_ids and item['id'] not in result,'UUID de versão ausente, inesperado ou duplicado')
        result[item['id']]=item
    require(set(result)==set(file_ids),'retorno /files/versions incompleto')
    return result

def reconcile_latest_versions(rows,fetcher=fetch_latest_versions):
    ids=[row['file_id'] for row in rows]
    details={}
    for start in range(0,len(ids),VERSION_BATCH_SIZE):
        batch=ids[start:start+VERSION_BATCH_SIZE]
        batch_details=fetcher(batch)
        require(isinstance(batch_details,dict) and set(batch_details)==set(batch),'lote de versões não reconciliado')
        details.update(batch_details)
    for row in rows:
        fid=row['file_id']; item=details[fid]
        require(item.get('id')==fid and item.get('latest_id')==fid,f'{fid}: latest ID não corresponde ao file_id atual')
        latest_version=str(item.get('latest_version') or item.get('version') or '').strip()
        require(re.fullmatch(r'[1-9][0-9]*',latest_version) is not None,f'{fid}: latest_version inválida')
        require(row.get('version') in (None,'',latest_version),f'{fid}: versão /files diverge de latest_version')
        latest_md5=item.get('latest_md5') or item.get('md5')
        latest_size=item.get('latest_size') or item.get('size')
        require(str(latest_md5).lower()==str(row['md5sum']).lower() and latest_size==row['file_size'],f'{fid}: checksum/tamanho de latest version diverge')
        releases=item.get('latest_release')
        require(isinstance(releases,list) and releases and all(isinstance(x,str) and RELEASE_RE.fullmatch(x) for x in releases),f'{fid}: latest_release ausente/inválido')
        releases=sorted(set(releases),key=_release_key)
        range_value=releases[0] if len(releases)==1 else f'{releases[0]} - {releases[-1]}'
        raw_release=row.get('data_release')
        if raw_release not in (None,''):
            normalized=re.sub(r'\s*-\s*','-',str(raw_release).strip())
            expected=re.sub(r'\s*-\s*','-',range_value)
            require(normalized==expected,f'{fid}: /files data_release diverge dos extremos latest_release')
        else:
            row['data_release']=range_value
        row['version']=latest_version
        row['available_releases']=releases
        row['version_metadata_source']='files_versions_api'
    return rows

def collect_two_stable_metadata_passes(page_size=200):
    first_meta,first_rows=metadata_pages(page_size=page_size,pass_label='gdc-pass-1')
    second_meta,second_rows=metadata_pages(page_size=page_size,pass_label='gdc-pass-2')
    require(first_meta['total_api']==second_meta['total_api'],'total GDC mudou entre passagens')
    require(first_meta['records_sha256']==second_meta['records_sha256'],'metadados GDC mudaram entre passagens')
    final_meta=dict(second_meta,metadata_passes=2,first_records_sha256=first_meta['records_sha256'],second_records_sha256=second_meta['records_sha256'],snapshot_stable=True)
    return final_meta,second_rows


## 4. Parsing estrito por caso, amostra e alíquota — fail-closed

**Uma linha por GDC file UUID** e **uma associação inequívoca** de caso, amostra e alíquota por arquivo. Se houver múltiplos vínculos ou identificadores faltantes, o arquivo é bloqueado e o manifesto exige reconciliação humana. Alíquotas múltiplas do mesmo paciente/condição são desempatas pelo barcode *antes* da análise, sem consultar expressão ou sobrevida.

In [ ]:
UUID_RE=re.compile(r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$')
MD5_RE=re.compile(r'^[0-9a-fA-F]{32}$')

def unique_one(values,what):
    values={str(x) for x in values if x not in (None,'')}
    if len(values)!=1: raise MetadataError(f'{what}: esperado identificador único; encontrados {len(values)}')
    return next(iter(values))

def unfold_file(record):
    file_id=record.get('file_id',record.get('id'))
    require(isinstance(file_id,str) and UUID_RE.fullmatch(file_id),'UUID GDC ausente/inválido')
    require(record.get('access')=='open' and record.get('state')=='released',
            f'Arquivo {file_id} não é open + released')
    require(record.get('data_category')=='Transcriptome Profiling' and
            record.get('data_type')=='Gene Expression Quantification','categoria/tipo inesperado')
    analysis=record.get('analysis') or {}
    require(analysis.get('workflow_type')==EXPECTED_WORKFLOW,'workflow inesperado')
    wf_version=analysis.get('workflow_version')
    require(bool(wf_version),'workflow_version ausente')
    size=record.get('file_size');require(type(size) is int and size>0,'tamanho inválido')
    md5=record.get('md5sum');require(isinstance(md5,str) and MD5_RE.fullmatch(md5) is not None,'MD5 inválido')
    cases=record.get('cases') or []
    case_id=unique_one([c.get('case_id') for c in cases],'case_id')
    patient=unique_one([c.get('submitter_id') for c in cases],'patient barcode')
    require(len(cases)==1,'arquivo associado a múltiplos casos')
    links=[]
    for c in cases:
        project=(c.get('project') or {}).get('project_id')
        require(project==PROJECT,'projeto diferente de TCGA-KIRC')
        for sample in c.get('samples') or []:
            for portion in sample.get('portions') or []:
                for analyte in portion.get('analytes') or []:
                    for aliquot in analyte.get('aliquots') or []:
                        links.append({'sample_id':sample.get('sample_id'),
                            'sample_barcode':sample.get('submitter_id'),
                            'sample_type':sample.get('sample_type'),
                            'aliquot_id':aliquot.get('aliquot_id'),
                            'aliquot_barcode':aliquot.get('submitter_id')})
    links=[dict(t) for t in {tuple(sorted(x.items())) for x in links}]
    require(len(links)==1,f'arquivo {file_id}: vínculos amostrais ausentes ou ambíguos')
    x=links[0]
    require(all(x.get(k) for k in ('sample_id','sample_type','aliquot_id','aliquot_barcode')),'metadados da alíquota incompletos')
    require(UUID_RE.fullmatch(str(x['sample_id'])) and UUID_RE.fullmatch(str(x['aliquot_id'])),'sample_id/aliquot_id inválidos')
    require(str(x['aliquot_barcode']).startswith(patient+'-'),'alíquota não corresponde ao paciente')
    require(x['sample_type'] in ('Primary Tumor','Solid Tissue Normal'),'tipo amostral não elegível')
    return dict(file_id=file_id,file_name=record.get('file_name'),file_size=size,md5sum=md5,
        access=record['access'],state=record['state'],workflow_version=wf_version,
        version=record.get('version'),data_release=record.get('data_release'),
        case_id=case_id,patient_id=patient,**x)

def build_manifest(metadata,records,version_fetcher=fetch_latest_versions):
    require(len(records)==metadata['total_api'],'páginas insuficientes')
    file_ids=[r.get('file_id',r.get('id')) for r in records]
    require(len(set(file_ids))==metadata['total_api'],'total API ≠ IDs únicos')
    selected=[];excluded=[];ambiguous=[]
    for rec in records:
        fid=rec.get('file_id',rec.get('id'))
        # Amostras não tumor/normal podem ser excluídas antes da verificação das ligações profundas
        st={s.get('sample_type') for c in (rec.get('cases') or []) for s in (c.get('samples') or [])}
        if st and not st.intersection({'Primary Tumor','Solid Tissue Normal'}):
            excluded.append({'file_id':fid,'reason':'sample_type não elegível'});continue
        try: selected.append(unfold_file(rec))
        except GateBlocked as e:
            ambiguous.append({'file_id':fid,'reason':str(e)})
    # Falha fechada: qualquer vínculo ambíguo ou desconhecido impede o GREEN do manifesto.
    require(not ambiguous,f'{len(ambiguous)} arquivos sem vínculo auditável; registrar e revisar')
    require(bool(selected),'nenhum arquivo elegível')
    # Seleção determinística por paciente/condição antes de qualquer expressão/desfecho.
    picked={}; duplicates=[]
    for row in sorted(selected,key=lambda r:(r['patient_id'],r['sample_type'],r['aliquot_barcode'],r['file_id'])):
        key=(row['patient_id'],row['sample_type'])
        if key in picked:
            duplicates.append({'file_id':row['file_id'],'reason':'alíquota elegível alternativa; seleção lexicográfica'})
        else: picked[key]=row
    chosen=list(picked.values())
    require(len({r['file_id'] for r in chosen})==len(chosen),'UUID duplicado após seleção')
    chosen=reconcile_latest_versions(chosen,fetcher=version_fetcher)
    authorized_exclusions=[]
    outlier_id='2b0106c5-7e32-423b-957f-06c9f2e7751b'
    outlier_workflow='5a02c12930eb23436322a7494047b907b97570c5'
    outlier=next((row for row in chosen if row['file_id']==outlier_id),None)
    if outlier is not None:
        require(outlier['workflow_version']==outlier_workflow,'workflow_version do UUID autorizado mudou; revisão humana necessária')
        alternatives=[row for row in selected if row['file_id']!=outlier_id and row['patient_id']==outlier['patient_id'] and row['sample_type']==outlier['sample_type']]
        require(not alternatives,'authorized_exclusion_alternative_candidate_present')
        rationale=('Exclusão metadata-only autorizada em 2026-10-21 exclusivamente por workflow_version divergente: '            +outlier_workflow+'. O data_release '+str(outlier['data_release'])+' se sobrepõe ao intervalo primário; '            +'faixa textual de release não é motivo de exclusão. Sem equivalência demonstrada entre workflows STAR Counts, '            +'misturá-los poderia confundir processamento com variação biológica. Não há arquivo alternativo para a mesma '            +'condição/paciente. Decisão baseada somente em metadados antes de expressão/desfecho; não afirma que o arquivo seja inválido.')
        outlier['selection_status']='excluded_workflow_version_incompatible'
        outlier['exclusion_reason']='operator_authorized_workflow_version_incompatibility'
        outlier['exclusion_rationale']=rationale
        authorized_exclusions.append({'file_id':outlier_id,'workflow_version':outlier['workflow_version'],
            'data_release':outlier['data_release'],'latest_release':outlier['available_releases'],
            'authorized_on':'2026-10-21','decision_basis':'workflow_version','rationale':rationale})
        excluded.append({'file_id':outlier_id,'reason':'operator_authorized_workflow_version_incompatibility','record':outlier})
        chosen=[row for row in chosen if row['file_id']!=outlier_id]
    require(bool(chosen),'nenhum arquivo primário após exclusões')
    wf={str(r['workflow_version']) for r in chosen}
    require(len(wf)==1,f'workflow_version heterogêneo após exclusão autorizada: {sorted(wf)}')
    release_sets=[set(row['available_releases']) for row in chosen]
    common_release=set.intersection(*release_sets)
    require(bool(common_release),'no_common_gdc_data_release')
    common_releases=sorted(common_release,key=_release_key)
    selected_common_release=max(common_releases,key=_release_key)
    total=validate_budget(chosen)
    manifest={'created_at_utc':timestamp(),'status':'VALIDATED_METADATA_NOT_DOWNLOADED',
        'download_eligible':True,'file_content_downloaded':False,'controlled_accessed':False,'analysis_executed':False,
        'query_audit':metadata,'selected_count':len(chosen),'selection_pre_patient_dedup':len(selected),
        'excluded_file_types_count':sum(1 for r in excluded if r.get('reason')=='sample_type não elegível'),
        'authorized_exclusions_count':len(authorized_exclusions),'alternative_aliquots_count':len(duplicates),
        'total_bytes':total,'max_bytes':MAX_AGGREGATE_BYTES,
        'workflow_version':next(iter(wf)),'data_release_ranges':sorted({str(r['data_release']) for r in chosen}),
        'common_data_releases':common_releases,'selected_common_data_release':selected_common_release,
        'records':chosen,'excluded':excluded,'authorized_exclusions':authorized_exclusions,'alternatives':duplicates}
    return manifest

In [ ]:
import copy

def fake_gdc_record():
    return {
      'file_id':'00000000-0000-0000-0000-000000000001','file_name':'fake.tsv',
      'file_size':200,'md5sum':'0'*32,'access':'open','state':'released',
      'data_category':'Transcriptome Profiling','data_type':'Gene Expression Quantification',
      'analysis':{'workflow_type':'STAR - Counts','workflow_version':'1.0'},
      'version':'1','data_release':'32.0 - 46.0',
      'cases':[{'case_id':'00000000-0000-0000-0000-000000000010',
                'submitter_id':'TCGA-AA-0001','project':{'project_id':'TCGA-KIRC'},
                'samples':[{'sample_type':'Primary Tumor',
                            'sample_id':'00000000-0000-0000-0000-000000000020',
                            'submitter_id':'TCGA-AA-0001-01A',
                            'portions':[{'analytes':[{'aliquots':[{
                               'aliquot_id':'00000000-0000-0000-0000-000000000030',
                               'submitter_id':'TCGA-AA-0001-01A-01R-0000-00'}]}]}]}]}]}

def synthetic_version_fetcher(records,release_by_id=None):
    by_id={row['file_id']:row for row in records}
    default_releases=['32.0','33.0','33.1','34.0','35.0','36.0','37.0','38.0','39.0','40.0','41.0','42.0','43.0','44.0','45.0','46.0']
    release_by_id=release_by_id or {}
    def fetch(file_ids):
        return {fid:{'id':fid,'latest_id':fid,'version':'1','latest_version':'1','md5':by_id[fid]['md5sum'],'size':by_id[fid]['file_size'],'latest_md5':by_id[fid]['md5sum'],'latest_size':by_id[fid]['file_size'],'latest_release':release_by_id.get(fid,default_releases)} for fid in file_ids}
    return fetch

def check_metadata_parser():
    good=fake_gdc_record()
    assert unfold_file(good)['sample_type']=='Primary Tumor'
    bad=copy.deepcopy(good);bad['access']='controlled'
    assert expect_red(unfold_file,bad)
    bad=copy.deepcopy(good);bad['cases'][0]['samples'][0]['portions'][0]['analytes'][0]['aliquots'].append({
        'aliquot_id':'00000000-0000-0000-0000-000000000031',
        'submitter_id':'TCGA-AA-0001-01A-01R-0001-00'})
    assert expect_red(unfold_file,bad)
    bad=copy.deepcopy(good);bad['md5sum']='bad'
    assert expect_red(unfold_file,bad)
    met={'total_api':1,'retrieved_utc':timestamp()}
    m=build_manifest(met,[good],version_fetcher=synthetic_version_fetcher([good]))
    assert m['total_bytes']==200 and m['selected_count']==1
    bad=copy.deepcopy(good);bad['data_release']=None
    recovered=build_manifest(met,[bad],version_fetcher=synthetic_version_fetcher([bad]))
    assert recovered['records'][0]['data_release']=='32.0 - 46.0'
    bad_version=copy.deepcopy(good);bad_version['version']='2'
    assert expect_red(build_manifest,met,[bad_version],version_fetcher=synthetic_version_fetcher([bad_version]))
    primary=copy.deepcopy(good);primary['file_id']='00000000-0000-0000-0000-000000000002';primary['analysis']['workflow_version']='5d8c131bbff59fb0c969217fc1d44e6d1503cd1f';primary['data_release']='32.0 - 46.0'
    outlier=copy.deepcopy(good);outlier['file_id']='2b0106c5-7e32-423b-957f-06c9f2e7751b';outlier['analysis']['workflow_version']='5a02c12930eb23436322a7494047b907b97570c5';outlier['data_release']='36.0 - 46.0';outlier['cases'][0]['case_id']='00000000-0000-0000-0000-000000000011';outlier['cases'][0]['submitter_id']='TCGA-AK-3444';outlier['cases'][0]['samples'][0]['sample_id']='00000000-0000-0000-0000-000000000021';outlier['cases'][0]['samples'][0]['submitter_id']='TCGA-AK-3444-01A';outlier['cases'][0]['samples'][0]['portions'][0]['analytes'][0]['aliquots'][0]['submitter_id']='TCGA-AK-3444-01A-02R-1325-07'
    records=[primary,outlier];metadata2={'total_api':2,'retrieved_utc':timestamp()}
    releases={'2b0106c5-7e32-423b-957f-06c9f2e7751b':['36.0','37.0','38.0','39.0','40.0','41.0','42.0','43.0','44.0','45.0','46.0']}
    compatible=build_manifest(metadata2,records,version_fetcher=synthetic_version_fetcher(records,releases))
    assert compatible['selected_count']==1 and compatible['common_data_releases'][-1]=='46.0'
    assert compatible['authorized_exclusions'][0]['decision_basis']=='workflow_version'
    overlap=copy.deepcopy(primary);overlap['file_id']='00000000-0000-0000-0000-000000000003';overlap['cases'][0]['case_id']='00000000-0000-0000-0000-000000000012';overlap['cases'][0]['submitter_id']='TCGA-CD-0002';overlap['cases'][0]['samples'][0]['sample_id']='00000000-0000-0000-0000-000000000022';overlap['cases'][0]['samples'][0]['submitter_id']='TCGA-CD-0002-01A';overlap['cases'][0]['samples'][0]['portions'][0]['analytes'][0]['aliquots'][0]['submitter_id']='TCGA-CD-0002-01A-01R-0000-00';overlap['data_release']='36.0 - 46.0'
    overlap_releases={overlap['file_id']:releases[outlier['file_id']]}
    overlap_manifest=build_manifest({'total_api':2,'retrieved_utc':timestamp()},[primary,overlap],version_fetcher=synthetic_version_fetcher([primary,overlap],overlap_releases))
    assert overlap_manifest['selected_count']==2 and overlap_manifest['common_data_releases'][0]=='36.0'
    disjoint_a=copy.deepcopy(primary);disjoint_a['file_id']='00000000-0000-0000-0000-000000000004';disjoint_a['data_release']='32.0 - 33.0'
    disjoint_b=copy.deepcopy(overlap);disjoint_b['file_id']='00000000-0000-0000-0000-000000000005';disjoint_b['data_release']='34.0 - 35.0'
    disjoint_releases={disjoint_a['file_id']:['32.0','33.0'],disjoint_b['file_id']:['34.0','35.0']}
    disjoint_rows=[disjoint_a,disjoint_b]
    assert expect_red(build_manifest,{'total_api':2,'retrieved_utc':timestamp()},disjoint_rows,version_fetcher=synthetic_version_fetcher(disjoint_rows,disjoint_releases))
    return True
assert check_metadata_parser()
write_json(ROOT/'tests'/'metadata_parser_smoke.json',{'passed':True,'scope':'synthetic-only','timestamp_utc':timestamp()})
print('Contraprovas de parser, workflow e interseção de releases: PASS (sintético).')

## 4.1. Auditoria adicional do contrato P1 — testes de paginação e falhas (sintéticos)

Antes da consulta externa, o notebook testa paginação, total reconciliado, acesso restrito e versão heterogênea por meio de uma API **simulada em memória**. Nenhum teste usa dados reais nem envia requisições. O relatório gerado é identificável como `synthetic_only`.


In [ ]:
# Esses testes não alteram consultas reais nem os estados dos gates P0–P4.
_real_gdc_get = gdc_get
try:
    rec1=fake_gdc_record()
    rec2=copy.deepcopy(rec1)
    rec2['file_id']='00000000-0000-0000-0000-000000000002'
    rec2['cases'][0]['case_id']='00000000-0000-0000-0000-000000000011'
    rec2['cases'][0]['submitter_id']='TCGA-AA-0002'
    rec2['cases'][0]['samples'][0]['sample_id']='00000000-0000-0000-0000-000000000021'
    rec2['cases'][0]['samples'][0]['submitter_id']='TCGA-AA-0002-01A'
    rec2['cases'][0]['samples'][0]['portions'][0]['analytes'][0]['aliquots'][0]['aliquot_id']='00000000-0000-0000-0000-000000000031'
    rec2['cases'][0]['samples'][0]['portions'][0]['analytes'][0]['aliquots'][0]['submitter_id']='TCGA-AA-0002-01A-01R-0000-00'
    # Reproduce the documented behavior: /files/_mapping indexes file fields,
    # but cases.* are fetched as related fields through /files?fields=...
    expected_keys={
       'files.file_id','files.file_name','files.access','files.state','files.data_category','files.data_type',
       'files.analysis.workflow_type','files.analysis.workflow_version',
       'files.file_size','files.md5sum'
    }
    assert verify_files_schema({'_mapping':{k:{} for k in expected_keys}})
    assert expect_red(verify_files_schema, {'_mapping':{k:{} for k in expected_keys-{'files.access'}}})
    assert expect_red(verify_files_schema, {'_mapping':{k:{} for k in expected_keys-{'files.file_id'}}})
    def _mock_get(path, params=None):
        if path=='/status': return {'data_release':'TEST_ONLY'}
        if path=='/files/_mapping': return {'_mapping':{k:{} for k in expected_keys}}
        if path=='/files':
            start=int(params['from']); size=int(params['size'])
            hits=[rec1,rec2][start:start+size]
            return {'data':{'hits':hits,'pagination':{'total':2,'from':start,'size':size,'count':len(hits),'sort':'file_id:asc'}},'warnings':{}}
        raise AssertionError('endpoint não esperado: '+path)
    gdc_get=_mock_get
    syn_meta,syn_records=collect_two_stable_metadata_passes(page_size=1)
    assert syn_meta['total_api']==2 and len(syn_meta['pages'])==2 and syn_meta['metadata_passes']==2
    assert build_manifest(syn_meta,syn_records,version_fetcher=synthetic_version_fetcher(syn_records))['selected_count']==2
    dup_rec=copy.deepcopy(rec2);dup_rec['file_id']=rec1['file_id']
    def _mock_bad(path,params=None):
        if path=='/files':
            start=int(params['from']);size=int(params['size'])
            hits=[rec1,dup_rec][start:start+size]
            return {'data':{'hits':hits,'pagination':{'total':2,'from':start,'size':size,'count':len(hits),'sort':'file_id:asc'}},'warnings':{}}
        return _mock_get(path,params)
    gdc_get=_mock_bad
    assert expect_red(metadata_pages,1)
    # Mesmo com protocolo de testes GREEN, nenhum gate clínico/real é liberado.
    write_json(ROOT/'tests'/'p1_contract_synthetic.json',{
      'timestamp_utc':timestamp(),'scope':'SYNTHETIC_ONLY','tests':6,
      'passed':6,'network_calls':0,'note':'Contraprovas de paginação e UUID não validam dados TCGA'})
    print('P1 pré-flight: 6 verificações sintéticas PASS; 0 consultas externas.')
finally:
    gdc_get=_real_gdc_get


## 5. Executar **somente** a consulta de metadados (opcional)

`CONSULTAR_GDC_METADATA=True` nesta R934.3 solicita a consulta real exclusivamente metadata-only. Manifesto completo e metadados brutos são armazenados na pasta privada do `RUN_ID` no Drive; o resumo público e o ZIP são gravados em `RUN_DIR/reports` e `RUN_DIR`. Nenhum dado de expressão é baixado. Falhas de schema, proveniência ou orçamento **não** são contornadas.

In [ ]:
GDC_MANIFEST=None
P1_PRELIMINARY_AUDIT={'time_utc':timestamp(),'mode':'METADATA_ONLY_NO_RNASEQ','status':'NOT_ATTEMPTED'}
if CONSULTAR_GDC_METADATA:
    try:
        metadata,records=collect_two_stable_metadata_passes(page_size=200)
        P1_PRELIMINARY_AUDIT.update(status='METADATA_RETRIEVED',api_file_count=metadata['total_api'],
               returned_rows=len(records),pages=len(metadata['pages']),query_sha256=metadata['query_sha256'],metadata_passes=metadata['metadata_passes'],snapshot_stable=metadata['snapshot_stable'])
        try:
            GDC_MANIFEST=build_manifest(metadata,records)
            manifest_path=write_json(ROOT/'private_metadata'/'gdc-manifest-R934.json',GDC_MANIFEST)
            manifest_sha256=hashlib.sha256(manifest_path.read_bytes()).hexdigest()
            manifest_path.with_suffix('.json.sha256').write_text(manifest_sha256+'\n',encoding='ascii')
            P1_PRELIMINARY_AUDIT.update(status='METADATA_VALIDATED_PRELIMINARY',
               selected_count=GDC_MANIFEST['selected_count'],
               selected_bytes=GDC_MANIFEST['total_bytes'],
               total_gib=round(GDC_MANIFEST['total_bytes']/1024**3,3),
               workflow_version=GDC_MANIFEST['workflow_version'],
               common_data_releases=GDC_MANIFEST['common_data_releases'],
               selected_common_data_release=GDC_MANIFEST['selected_common_data_release'],
               authorized_exclusions_count=GDC_MANIFEST['authorized_exclusions_count'],
               metadata_manifest_sha256=manifest_sha256)
            record_stage('gdc_metadata_manifest','preliminary_validated',{'selected_count':GDC_MANIFEST['selected_count'],'selected_bytes':GDC_MANIFEST['total_bytes'],'manifest_sha256':manifest_sha256})
            print('Metadados verificados preliminarmente; NENHUM DOWNLOAD FEITO.')
            print('Atenção: P0/P1 não são automaticamente promovidos a GREEN.')
        except GateBlocked as e:
            P1_PRELIMINARY_AUDIT.update(status='BLOCKED_MANIFEST',error_type=type(e).__name__,details_redacted=True)
            write_json(ROOT/'private_metadata'/'P1_blocker_private.json',{'time_utc':timestamp(),'error_type':type(e).__name__,'details':str(e)})
            record_stage('gdc_metadata_manifest','blocked',{'error_type':type(e).__name__,'details_redacted':True})
            print('Manifesto bloqueado (fail closed); causa detalhada preservada somente em private_metadata.')
    except Exception as e:
        P1_PRELIMINARY_AUDIT.update(status='BLOCKED_API_OR_SCHEMA',error_type=type(e).__name__,details_redacted=True)
        write_json(ROOT/'private_metadata'/'P1_query_error_private.json',{'time_utc':timestamp(),'error_type':type(e).__name__,'details':str(e)})
        record_stage('gdc_metadata_query','blocked',{'error_type':type(e).__name__,'details_redacted':True})
        print('Consulta metadata-only bloqueada; detalhes privados preservados em private_metadata.')
else:
    print('Consulta de metadados desativada.')
P1_PRELIMINARY_AUDIT.update(run_id=RUN_ID,notebook='R934.3',file_content_downloaded=False,analysis_executed=False)
write_json(ROOT/'reports'/'P1_metadata_summary_PUBLIC.json',P1_PRELIMINARY_AUDIT)
print('Resumo P1 (sem identificadores):',json.dumps(P1_PRELIMINARY_AUDIT,ensure_ascii=False,indent=2))


### Leitura do relatório P1

O arquivo `reports/P1_metadata_summary_PUBLIC.json` contém apenas estatísticas agregadas e causas de bloqueio. Se o estado for `METADATA_VALIDATED_PRELIMINARY`, os metadados passaram pela verificação preliminar, **não** houve download de RNA-seq, e P0/P1 continuam sujeitos à revisão científica. Se houver `BLOCKED_*`, corrija a causa, sem remover filtros open-only, limitar amostras silenciosamente ou mudar o teto de 6 GiB.


In [ ]:
assert EXECUTAR_DOWNLOADS is False, 'CONFIGURAÇÃO PROIBIDA: downloads não autorizados na R934.2'
assert P1_PRELIMINARY_AUDIT.get('mode')=='METADATA_ONLY_NO_RNASEQ'
write_json(ROOT/'reports'/'p0_independent_review_REQUIRED.json',{
 'status':'PENDING_INDEPENDENT_REVIEW',
 'source_notebook':'R934.1',
 'requirements':[
    'parecer metodológico registrado com SHA-256',
    'auditoria das fontes biológicas do painel predefinido',
    'reconciliação de version/data_release/workflow GDC',
    'demonstração da extração inequívoca de aliquot_id por file_id',
    'pipeline R nested-CV com VST e size factors congelados',
    'plano de comparação clínica e IC da diferença AUC/IPCW'
 ],
 'guardrails':{'downloads_enabled':False,'gates_auto_promoted':False,
               'threshold_bytes':MAX_AGGREGATE_BYTES},'time_utc':timestamp()
})
print('R934.2: downloads bloqueados; próximo gate permanece pendente de revisão.')


## 6. Selo dos gates P0–P4 e registro da revisão humana

Os estados abaixo não podem ser promovidos por resultados biológicos, número de genes ativos, p-valores, AUC ou HR. P0 só será GREEN após um aceite metodológico **registrado pelo pesquisador** e verificação independente dos testes. P1 depende de manifesto validado, compatibilidade de workflow/release e confirmação humana de que o release comum está vigente. A aprovação dos metadados **não** comprova arquivos obtidos.

In [ ]:
# O aceite metodológico precisa de documento externo e hash, não apenas deste botão/variável.
REVISAO_REGISTRADA=False
REVISAO_DOCUMENTO_SHA256=''  # 64 caracteres hex de parecer metodológico revisado
RELEASE_VIGENTE_CONFERIDO=False  # comparado ao release GDC atual e documentado por revisor
P2_DOWNLOADS_VERIFICADOS=False
P3_PIPELINE_NESTED_VST_VERIFICADO=False
P4_ANALISE_REVISADA=False

def gates_snapshot():
    accepted=bool(REVISAO_REGISTRADA and ACEITE_METODOLOGICO.strip() and
                  re.fullmatch('[0-9a-fA-F]{64}',REVISAO_DOCUMENTO_SHA256))
    states={
      'P0': bool(TDD_GREEN and accepted and PANEL_APPROVED),
      'P1': bool(GDC_MANIFEST is not None and GDC_MANIFEST.get('status')=='VALIDATED_METADATA_NOT_DOWNLOADED' and RELEASE_VIGENTE_CONFERIDO),
      'P2': bool(P2_DOWNLOADS_VERIFICADOS),
      'P3': bool(P3_PIPELINE_NESTED_VST_VERIFICADO),
      'P4': bool(P4_ANALISE_REVISADA)
    }
    # Sequencial: não há GREEN posterior enquanto o anterior é RED.
    for i in range(1,5):
        k=f'P{i}'
        states[k]=bool(states[k] and states[f'P{i-1}'])
    return states
GATES=gates_snapshot()
write_json(ROOT/'reports'/'gates_R934.json',{'time_utc':timestamp(),'spec_hash':SPEC_HASH,'gates':GATES,
    'note':'TDD sintético != validação de dados TCGA; revisão independente exigida'})
print('Gates:',GATES)

## 7. P2 — downloads deliberadamente não implementados nesta versão

R934.3 é exclusivamente metadata-only. Esta versão não contém downloader nem chamada ao endpoint `/data`; P2 permanece RED. Qualquer transferência exigirá outra spec, pipeline verificado, revisão P0/P1 e autorização/execução separadas.

O operador instruiu `EXECUTAR_DOWNLOADS=False`; a flag fica fixada nesta versão e não pode ser habilitada por configuração notebook.

In [ ]:
assert EXECUTAR_DOWNLOADS is False, 'R934.3 é metadata-only; P2 permanece bloqueado'
P2_DOWNLOADS_VERIFICADOS=False
record_stage('P2_content_download','blocked_by_spec',{'file_content_downloaded':False})
print('P2 bloqueado pela especificação R934.3; nenhum arquivo RNA-seq foi baixado.')

## 8. P3 — desenho prognóstico e funções de proteção (não analisar dados TCGA ainda)

**Unidade = paciente.** Estratificar por evento (70% descoberta/30% holdout) com seed 2026. Registrar IDs e somente seus hashes em relatórios públicos. Não usar dados da validação para escolher genes, regularização, z-score, dispersão ou cutoff. PCK1/PCK2 pertencem à lista separada de gliconeogênese. Seleção LASSO deve usar CV aninhada 5×5, 5 repetições no discovery, e `lambda.1se` final, sem fallback para `lambda.min` em caso de modelo vazio.

O pipeline estatístico definitivo demanda **integração versionada R/Bioconductor (DESeq2/VST), `glmnet`/`survival` e `timeROC`** depois que P2 estiver completo. Este notebook implementa validação de entrada e partição sem executar modelos reais, evitando dar aparência de cálculo clínico.

In [ ]:
def make_stratified_split(records,seed=SEED):
    """records: lista de dicts com patient_id, event (0/1); depende de sklearn no Colab."""
    from sklearn.model_selection import train_test_split
    ids=[r['patient_id'] for r in records]
    ev=[r['event'] for r in records]
    require(len(ids)==len(set(ids)),'pacientes repetidos antes do particionamento')
    require(len(ids)>20 and set(ev)=={0,1},'amostra/eventos insuficientes')
    require(min(Counter(ev).values())>=10,'estratos com poucos pacientes')
    train,test=train_test_split(records,test_size=.30,random_state=seed,stratify=ev)
    validate_patient_partition([x['patient_id'] for x in train],[x['patient_id'] for x in test])
    return train,test

def stratified_fold_ids(records,k=5,seed=SEED):
    from sklearn.model_selection import StratifiedKFold
    events=[r['event'] for r in records]
    require(min(Counter(events).values())>=k,'evento/censura insuficiente para folds')
    splitter=StratifiedKFold(n_splits=k,shuffle=True,random_state=seed)
    ids={}
    for fold,(tr,va) in enumerate(splitter.split(range(len(records)),events)):
        require(bool(sum(events[i] for i in tr)) and bool(sum(events[i] for i in va)),
                f'fold {fold} sem eventos')
        for i in va: ids[records[i]['patient_id']]=fold
    require(len(ids)==len(records),'folds não cobriram todas as pessoas')
    return ids

# DEMONSTRAÇÃO SINTÉTICA, sem expressão humana e sem resultados de sobrevivência.
synthetic=[{'patient_id':f'FAKE-{i:03d}','event':int(i%3==0)} for i in range(150)]
train_demo,holdout_demo=make_stratified_split(synthetic)
folds_demo=stratified_fold_ids(train_demo,5,SEED)
assert len(train_demo)+len(holdout_demo)==150
write_json(ROOT/'tests'/'demo_synthetic_split.json',{'seed':SEED,'train_n':len(train_demo),
   'holdout_n':len(holdout_demo),'train_event_n':sum(r['event'] for r in train_demo),
   'holdout_event_n':sum(r['event'] for r in holdout_demo),
   'fold_counts':dict(Counter(folds_demo.values())),'note':'DADOS SINTÉTICOS, NÃO TCGA'})
print('Demonstração sintética do split OK. Nenhuma análise prognóstica real executada.')

### Especificação operacional do módulo R a desenvolver antes do P3 GREEN

**Entrada:** um *manifesto P2 verificado* e apenas dados open-access necessários. `patient_id` interno, contagens não negativas inteiras com Ensembl ID + gene annotation, tabela clínica de OS/idade/sexo/estágio. Para contagens duplicadas da mesma entidade, somar inteiros; nunca `limma::avereps`. Separar tumores primários (prognóstico) e normais (MUI descritivo) e registrar inclusão/exclusão. Descartar tempos OS ausentes/não positivos mediante fluxograma; não impor 30 dias sem justificativa.

**Pré-processamento:** filtros de CPM e geometria/size factors aprendidos no discovery (ou no treino do fold), dispersão VST e escala congeladas. Para holdout, não executar estimadores distributivos novos. **Modelagem:** 5 folds externos × 5 repetições; para cada outer-train, fazer 5 folds internos, com todas as transformações ajustadas só no train interno; a agregação out-of-fold descreve estabilidade, não seleciona o modelo final. Depois fazer CV 5-fold no discovery completo, escolher `lambda.1se` e treinar modelo clínico e ampliado. Se seleção resultar vazia, registrar isso sem trocar arbitrariamente de lambda ou genes.

**Avaliação:** holdout aberto uma vez; AUC cumulative/dynamic IPCW em 60 meses (se ≥10 eventos até 60 meses, censura Ĝ(60)≥0,10 e suporte de seguimento), bootstrap pareado B=1000 com ≥90% reamostragens estimáveis. O modelo de censura deve usar o discovery como referência, sem estimar IPW no holdout para o preditor. Reportar C-index, Brier e calibração separadamente. Sem resultados desejados como gates.

**Ressalva:** esta seção é especificação executável a implementar/validar no módulo R, não uma análise executada. P3 permanece RED.

## 9. Predição genômica (Genomic Intelligence) — evidência separada

Mantenha as predições preliminares dos genes **SLC2A1** e **SDHB** como resultados computacionais exploratórios, sem vinculá-los ao risco clínico. Os resultados do plugin são distintos do GDC e não serão usados na seleção de características a partir do holdout. O script inicial do estudo contém números não verificados; não incorporá-los às tabelas finais.

In [ ]:
GI_PILOT=[{'gene':'SLC2A1','ensembl_id':'ENSG00000117394','pred_promoter_windows':13,
    'pred_tpm':39.9449,'reference':'Ensembl/GRCh38','type':'in_silico_not_TCGA'},
   {'gene':'SDHB','ensembl_id':'ENSG00000117118','pred_promoter_windows':7,
    'pred_tpm':126.9445,'reference':'Ensembl/GRCh38','type':'in_silico_not_TCGA'}]
write_json(ROOT/'reports'/'genomic_intelligence_exploratory.json',{
 'source':'Genomic Intelligence plugin, execução preliminar na conversa anterior',
 'limits':'Não representa diferença tumor-normal, coorte TCGA ou validação clínica',
 'results':GI_PILOT})
print('Genomic Intelligence: predições isoladas do pipeline de prognóstico.')

## 10. Auditoria, integridade e exportação pública versionada no Drive

Gera hashes de relatórios **sem incluir dados individuais**. `private_metadata/` contém IDs e respostas brutas; a pasta private_metadata não entra no ZIP público. Cada execução usa `RUN_DIR` UTC exclusivo no Drive; nenhum run anterior é sobrescrito. O ZIP contém somente `tests/`, `reports/` e `logs/` agregados.

In [ ]:
import zipfile
public_summary_path=ROOT/'reports'/'P1_metadata_summary_PUBLIC.json'
if public_summary_path.is_file():
    summary_sha256=hashlib.sha256(public_summary_path.read_bytes()).hexdigest()
    public_summary_path.with_suffix('.json.sha256').write_text(summary_sha256+'\n',encoding='ascii')
PUBLIC_DIRS=['tests','reports','logs']
public_files=[p for folder in PUBLIC_DIRS for p in (ROOT/folder).rglob('*') if p.is_file()]
checksum=[{'path':str(p.relative_to(ROOT)),'size':p.stat().st_size,
    'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in public_files]
write_json(ROOT/'reports'/'public_artifact_checksums.json',checksum)
zip_path=ROOT/'R934_auditoria_publica.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in PUBLIC_DIRS:
        for p in (ROOT/folder).rglob('*'):
            if p.is_file(): zf.write(p,p.relative_to(ROOT))
zip_sha256=hashlib.sha256(zip_path.read_bytes()).hexdigest()
zip_sidecar=ROOT/'R934_auditoria_publica.zip.sha256'
zip_sidecar.write_text(zip_sha256+'\n',encoding='ascii')
private_index_path=ROOT/'private_metadata'/'run_artifact_index.json'
private_artifacts=[p for p in ROOT.rglob('*') if p.is_file() and p not in (private_index_path,private_index_path.with_suffix('.json.sha256'))]
private_index={'run_id':RUN_ID,'created_at_utc':timestamp(),'artifact_count':len(private_artifacts),'artifacts':[{'path':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(private_artifacts)],'private_metadata_in_public_zip':False}
write_json(private_index_path,private_index)
private_index_digest=hashlib.sha256(private_index_path.read_bytes()).hexdigest()
private_index_path.with_suffix('.json.sha256').write_text(private_index_digest+'\n',encoding='ascii')
print('ZIP público versionado no Drive:',zip_path,'| SHA-256:',zip_sha256)
print('Estado ao encerrar:',gates_snapshot())

## 11. Limites da implementação e referências

**Executável nesta R934.3 no Colab:** contraprovas sintéticas, consulta `/status`, `/files/_mapping`, paginação `/files` e `/files/versions` metadata-only, manifesto privado versionado, resumo e ZIP públicos agregados. `EXECUTAR_DOWNLOADS=False`; não há downloader nesta versão.

**Ainda não executado e não aprovado:** download de contagens, auditoria clínica, transformação VST/R, nested LASSO-Cox, AUC com IC 95%, Brier, nomograma/calibração, análises imunes/ssGSEA e validação externa. P2–P4 permanecem RED até a implementação estatística efetiva e a revisão humana. A AUC 0,79 vs 0,69 ou outros números antecipados no manuscrito não são resultados observados.

**Documentação consultável:**
- National Cancer Institute, GDC API — [Getting Started](https://docs.gdc.cancer.gov/API/Users_Guide/Getting_Started/) e [System Information](https://docs.gdc.cancer.gov/API/Users_Guide/System_Information/).
- GDC Search API: [Search and Retrieval](https://docs.gdc.cancer.gov/API/Users_Guide/Search_and_Retrieval/).
- Scikit-survival (para futura avaliação IPCW): [cumulative_dynamic_auc](https://scikit-survival.readthedocs.io/en/stable/api/generated/sksurv.metrics.cumulative_dynamic_auc.html).
- Documento fornecido: *Protocolo pré-análise R934 — ccRCC / TCGA-KIRC*, 21/10/2026 (base principal). Entendimentos divergentes do documento anterior foram indicados explicitamente neste caderno.

**Segurança:** pesquisa retrospectiva exploratória; não é dispositivo médico nem orientação individual de tratamento. A consulta real nesta versão lê somente metadados open/released; nenhum arquivo RNA-seq é transferido. O limite de 6 GiB e as autorizações não promovem P0–P4 automaticamente.